# Termination Criteria

The [termination criteria](https://github.com/BioSystemsUM/ReactEA/blob/main/src/reactea/optimization/jmetal/terminators.py) stop the EAs.

By default, a run stops after `generations` generations or, if a `patience` argument is specified in the configuration file, earlier if the mean score of the population does not improve for `patience` generations.

Other criteria are passed to `run_reactea` with the `termination_criterion` argument.

## Built-in criteria

| Criterion | Stops when |
|---|---|
| `StoppingByEvaluations(max_generations)` | the maximum number of generations is reached |
| `StoppingByEvaluationsOrImprovement(patience)` | the mean score does not improve, in any objective, for `patience` generations |
| `StoppingByEvaluationsOrMeanFitnessValue(expected_value)` | the mean score reaches `expected_value` in every objective |
| `StoppingByScore(value, objective=0)` | a solution reaches `value` in an objective |

All criteria also stop after a maximum number of generations: `max_generations` if given, otherwise `generations` from the configuration file.

In [ ]:
from reactea.optimization.jmetal.terminators import StoppingByScore

# stop when a solution reaches a score of 0.9 in the first objective (or after `generations` generations)
criterion = StoppingByScore(0.9, objective=0)

## Example of how to implement your own termination criterion

Termination criteria inherit from the `ReactEATerminationCriterion` class and implement the `stop` method.

At each generation, `stop` receives the scores of the current solutions (one list per solution, with one value per objective, as returned by the evaluation functions) and whether each objective is maximized. If it returns `True`, the EA stops.

The base class counts the generations and stops the EA when the maximum number of generations is reached.

In [ ]:
from reactea.optimization.jmetal.terminators import ReactEATerminationCriterion


class StopWhenEnoughGoodSolutions(ReactEATerminationCriterion):
    """Stops when n solutions reach a minimum score in every (maximized) objective."""

    def __init__(self, n: int, min_score: float, max_generations: int = None):
        super().__init__(max_generations)
        self.n = n
        self.min_score = min_score

    def stop(self, scores, maximize):
        good = [s for s in scores if all(value >= self.min_score for value in s)]
        return len(good) >= self.n


criterion = StopWhenEnoughGoodSolutions(n=3, min_score=0.8)

## Using a termination criterion

```python
from reactea import run_reactea

run_reactea('config.yaml', case_study, termination_criterion=StopWhenEnoughGoodSolutions(n=3, min_score=0.8))
```

A criterion keeps the state of a run (e.g. the number of generations), so create a new one for each run.